In [58]:
# Estas lineas evitan la necesidad de reiniciar manualmente el kernel o recargar módulos importados cada vez que realizas un cambio en el código
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [59]:
import random
import pandas as pd
import numpy as np
import matplotlib as plt

In [60]:
from src.utils import print_dataset_extract
from src.preprocessing import fix_data

# Analisis exploratorio y preprocesamiento

## 1.1

In [61]:
df_dev = pd.read_csv('data/raw/rendimiento_estudiantes_dev.csv')
df_test = pd.read_csv('data/raw/rendimiento_estudiantes_test.csv')

In [62]:
print_dataset_extract(df_dev)

      horas_estudio  asistencia  nota_previa  horas_sueno  participacion  \
1006            4.6      42.800          4.2          5.7            2.5   
1007            8.9      53.800          5.6         10.2            6.1   
1008            7.5      64.600          7.8          NaN            6.7   
1009            7.7      80.800          4.0          7.3            3.9   
1010            7.4      71.400          5.1          9.3            6.6   
1011           11.5       0.609          8.2          4.9            7.0   
1012            1.1      33.900          6.9          7.2            6.6   

      horas_extracurricular  acceso_internet  distancia_escuela_km  \
1006                    1.0                1                   5.8   
1007                    2.2                0                   1.7   
1008                    2.6                1                   3.4   
1009                    5.4                0                   1.6   
1010                    0.7              

In [63]:
df_dev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5058 entries, 0 to 5057
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   horas_estudio          4783 non-null   float64
 1   asistencia             5058 non-null   float64
 2   nota_previa            4530 non-null   float64
 3   horas_sueno            4785 non-null   float64
 4   participacion          4878 non-null   float64
 5   horas_extracurricular  5058 non-null   float64
 6   acceso_internet        5058 non-null   int64  
 7   distancia_escuela_km   5058 non-null   float64
 8   nivel_socioeconomico   4966 non-null   float64
 9   tamano_clase           5058 non-null   int64  
 10  escuela                5058 non-null   object 
 11  semestre               5058 non-null   object 
 12  rendimiento            5058 non-null   object 
dtypes: float64(8), int64(2), object(3)
memory usage: 513.8+ KB


In [64]:
print(df_dev["rendimiento"].value_counts())
print()
print(df_dev["escuela"].value_counts())
print()
print(df_dev["semestre"].value_counts())

rendimiento
Excelente       1775
Bueno           1329
Regular         1136
Insuficiente     818
Name: count, dtype: int64

escuela
G    813
D    788
B    666
E    619
A    592
F    546
H    522
C    463
e     12
g      8
h      7
b      7
f      5
d      5
a      4
c      1
Name: count, dtype: int64

semestre
2022-1    879
2022-2    866
2024-2    847
2023-1    831
2023-2    820
2024-1    815
Name: count, dtype: int64


Observing some stats from the dataset, it is visible that some features are categorical, such as "rendimiento", "escuela" and "semestre". <br> For the first one, there are four types: Excelente, Bueno, Regular and Insuficiente. For multiclass classification I choose to represent those categories with 3, 2, 1, 0 respectivelly (there is a clear order, we suppose the distance between the labels is the same). Instead, for binary classification we must group Excelente, Bueno and Regular in "Pass" (1), opposed to Insuficiente, "Fail" (0). <br> We can see in "escuela" there are 8 different options: from a to h, I choose to do One hot encoding (we must convert the upper cases in lower cases before). <br> For "semestre" there is a progression, an order, There are 6 options so, in chronological order I assign an index to each time period: 2022-1 is 1, 2022-2 is 2, 2023-1 is 3, etc...

In [65]:
print(df_dev.isna().mean())

horas_estudio            0.054369
asistencia               0.000000
nota_previa              0.104389
horas_sueno              0.053974
participacion            0.035587
horas_extracurricular    0.000000
acceso_internet          0.000000
distancia_escuela_km     0.000000
nivel_socioeconomico     0.018189
tamano_clase             0.000000
escuela                  0.000000
semestre                 0.000000
rendimiento              0.000000
dtype: float64


In [66]:
import pandas as pd
import numpy as np

# 1. Ver qué columnas tienen NaN y cuántos
df_dev.isnull().sum()[df_dev.isnull().sum() > 0]

# 2. Crear indicadoras de missingness
for col in df_dev.columns[df_dev.isnull().any()]:
    df_dev[f'{col}_is_nan'] = df_dev[col].isnull().astype(int)

# 3. Para cada indicadora, analizar vs las otras features
# Si la otra feature es categórica:
df_dev.groupby('escuela')['nota_previa'].mean().sort_values(ascending=False)

# Si la otra feature es numérica:
df_dev.groupby('nota_previa')['semestre'].mean()

TypeError: agg function failed [how->mean,dtype->object]